In [7]:
import os 
import sys
sys.path.append('/home/hb-nano/mirsaid/face-recognition')
from ultralytics import YOLO
import cv2

data_images = '/home/hb-nano/mirsaid/face-recognition/data/ilhan'
model = YOLO('/home/hb-nano/mirsaid/face-recognition/yolov8m-face.pt')

for filename in os.listdir(data_images):
    image = cv2.imread(os.path.join(data_images, filename))

    # Detect and crop face
    yolo_results = model(image, max_det=1, verbose=False)

    x1, y1, x2, y2, _, _ = yolo_results[0].boxes.data[0].cpu().numpy()

    face_image = image[int(y1):int(y2), int(x1):int(x2)]

    # save image to the folder
    cv2.imwrite(f'/home/hb-nano/mirsaid/face-recognition/data/ilhan_2/{filename}.jpg', face_image)

In [1]:
import cv2
import pandas as pd
import os
import random

# Full Paths
gt_path = '/home/hbvision/mirsaid/smart-office/output_mot_11.txt' # path to the ground truth file
video_path = '/home/hbvision/mirsaid/smart-office/output_11_processed_fps.mp4'

# Load ground truth data
gt_data = pd.read_csv(gt_path, header=None)
gt_data.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'class', 'visibility']

# Sort by frame number and filter visible objects
gt_data = gt_data.sort_values(by='frame')

# Assign a random color to each object ID
id_colors = {}
unique_ids = gt_data['id'].unique()
for obj_id in unique_ids:
    id_colors[obj_id] = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))


cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('output.avi', cv2.VideoWriter_fourcc(*'XVID'), 30, (width, height))


frame_num = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    frame_data = gt_data[gt_data['frame'] == frame_num]

    for _, row in frame_data.iterrows():
        if row['class'] != 1:
         continue

        x, y, w, h = int(row['x']), int(row['y']), int(row['w']), int(row['h'])
        obj_id = int(row['id'])
        visibility = row['visibility']

        # Get color for the current object ID
        color = id_colors[obj_id]

        # Draw bounding box
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

        # Add text for ID and visibility level
        text = f'ID: {obj_id}, Vis: {visibility:.2f}'
        cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    out.write(frame)
    cv2.imshow('Tracking', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
out.release()
cv2.destroyAllWindows()
